# Bitcoin Naive Persistence Audit

## Role
This notebook independently verifies the strongest and simplest Bitcoin benchmark: the frozen `Naive` forecast column.

## Inputs
The canonical Bitcoin daily target series and the frozen `Naive` forecast column from `results/validated_forecasts.csv`.

## Outputs
An independently reconstructed Naive vector, a row-level comparison, exact whole-vector equality evidence, independently recomputed metrics, and boundary/leakage checks.

## Depends On
`01_Bitcoin_Data_EDA.ipynb`
`07_Bitcoin_Forecast_Freeze_and_Validation.ipynb`

## Authoritative Status
`INDEPENDENT BASELINE AUDIT`

## What This Notebook Does Not Do
It does not generate a new authoritative forecast, does not audit the other nine models at this same reconstruction level, does not redefine the train/test split, does not perform final statistical inference, and does not calculate robustness or Trust Scores.

# 1. Objective and Audit Question

> Does the frozen `Naive` forecast vector genuinely implement pure one-step persistence -- the previous observed actual value and nothing else -- with no same-day target use, no index shift, no boundary error, and no hidden construction bug?

This matters because the project repeatedly finds that Naive is difficult for more complex systems to beat (Notebooks 02-07). Therefore the Naive result must be independently verified rather than merely trusted because it appears in a saved CSV.

> This notebook intentionally re-derives the Naive forecast from the canonical target using an independent code path and compares it against the frozen vector row by row.

# 2. Setup

Standard project-root discovery. Only the canonical target, the frozen forecast matrix, and the repository's canonical metric functions are loaded -- no model-training library is imported, and this notebook writes no result artifact.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.bitcoin_pipeline import *

RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False

# 3. Data and Method

## 3.1 Audit Strategy

The manual Naive reconstruction is `y_hat_t = y_(t-1)`: for the first final-test target, the final training observation; for every later final-test target, the immediately preceding observed actual.

The manual forecast is generated from the canonical `Actual` series only. It does not read the stored `Naive` forecast values at any point during construction. Only after the manual vector is complete is it compared with the frozen vector.

> Two independent derivation paths converging on numerically identical vectors provide evidence that the stored Naive forecast was constructed correctly.

This is not a cryptographic proof. It is an independent numerical reconstruction audit.

## 3.2 Canonical Target and Frozen Naive Sources

In [2]:
# Load the canonical split and frozen forecast matrix.
_, target = load_bitcoin_target(ROOT)
train, test = canonical_split(target)
validated = load_validated_forecasts(ROOT)

sources_table = pd.DataFrame({
    'Source': [
        'src.bitcoin_pipeline.load_bitcoin_target(ROOT)',
        str(train.index.max().date()),
        str(test.index.min().date()),
        str(test.index.max().date()),
        len(test),
        'results/validated_forecasts.csv -> Naive',
        'Reconstructed independently in this notebook (Section 4)',
    ],
}, index=[
    'Canonical target series', 'Final train date', 'Final test start', 'Final test end',
    'Final test observations', 'Frozen Naive vector', 'Manual Naive vector',
])
sources_table

,Source
Canonical target series,src.bitcoin_pipeline.load_bitcoin_target(ROOT)
Final train date,2023-08-11
Final test start,2023-08-12
Final test end,2026-07-07
Final test observations,1061
Frozen Naive vector,results/validated_forecasts.csv -> Naive
Manual Naive vector,Reconstructed independently in this notebook (...


# 4. Row-Level Audit

## 4.1 Leading and Trailing Rows

| Column | Meaning |
|---|---|
| Actual | Actual target for the forecasted date |
| Previous Actual | Most recently available observed actual before the forecast date |
| Manual Naive | Independently reconstructed persistence forecast |
| Frozen Naive | Saved authoritative Naive forecast |

Matching `Previous Actual == Manual Naive` demonstrates the reconstruction rule was applied correctly; matching `Manual Naive == Frozen Naive` demonstrates agreement with the stored artifact for these specific rows. These displayed rows alone do not prove whole-series identity -- that is established in Section 4.2.

In [3]:
# Reconstruct persistence directly from the previous observed actual --
# built only from the canonical target, never from the stored Naive column.
naive = target.shift(1).reindex(test.index)

rows = pd.DataFrame({
    'Forecast date': test.index,
    'Actual': test.to_numpy(),
    'Previous Actual': [train.iloc[-1], *test.iloc[:-1]],
    'Manual Naive': naive.to_numpy(),
    'Frozen Naive': validated.Naive.to_numpy(),
})

display(rows.head(10))
display(rows.tail(10))

,Forecast date,Actual,Previous Actual,Manual Naive,Frozen Naive
0,2023-08-12 00:00:00+00:00,29415.0,29398.0,29398.0,29398.0
1,2023-08-13 00:00:00+00:00,29284.0,29415.0,29415.0,29415.0
2,2023-08-14 00:00:00+00:00,29408.0,29284.0,29284.0,29284.0
3,2023-08-15 00:00:00+00:00,29172.0,29408.0,29408.0,29408.0
4,2023-08-16 00:00:00+00:00,28701.0,29172.0,29172.0,29172.0
5,2023-08-17 00:00:00+00:00,26642.0,28701.0,28701.0,28701.0
6,2023-08-18 00:00:00+00:00,26051.0,26642.0,26642.0,26642.0
7,2023-08-19 00:00:00+00:00,26097.0,26051.0,26051.0,26051.0
8,2023-08-20 00:00:00+00:00,26192.0,26097.0,26097.0,26097.0
9,2023-08-21 00:00:00+00:00,26125.0,26192.0,26192.0,26192.0


,Forecast date,Actual,Previous Actual,Manual Naive,Frozen Naive
1051,2026-06-28 00:00:00+00:00,59473.29,59940.07,59940.07,59940.07
1052,2026-06-29 00:00:00+00:00,60163.86,59473.29,59473.29,59473.29
1053,2026-06-30 00:00:00+00:00,58526.17,60163.86,60163.86,60163.86
1054,2026-07-01 00:00:00+00:00,59963.46,58526.17,58526.17,58526.17
1055,2026-07-02 00:00:00+00:00,61479.10,59963.46,59963.46,59963.46
1056,2026-07-03 00:00:00+00:00,62522.46,61479.10,61479.10,61479.10
1057,2026-07-04 00:00:00+00:00,63086.18,62522.46,62522.46,62522.46
1058,2026-07-05 00:00:00+00:00,63587.06,63086.18,63086.18,63086.18
1059,2026-07-06 00:00:00+00:00,64000.10,63587.06,63587.06,63587.06
1060,2026-07-07 00:00:00+00:00,63902.77,64000.10,64000.10,64000.10


## 4.2 Full-Series Match Summary

This is the whole-series proof, computed explicitly rather than left hidden inside a single assertion.

In [4]:
manual_values = naive.to_numpy()
frozen_values = validated.Naive.to_numpy()
diff = manual_values - frozen_values
exact_matches = int((manual_values == frozen_values).sum())

match_summary = pd.DataFrame({
    'Result': [
        len(naive),
        exact_matches,
        len(naive) - exact_matches,
        float(np.max(np.abs(diff))),
        float(np.mean(np.abs(diff))),
        'PASS' if np.array_equal(manual_values, frozen_values) else 'FAIL',
        'PASS' if np.allclose(manual_values, frozen_values) else 'FAIL',
    ],
}, index=[
    'Total rows', 'Exact matching rows', 'Mismatching rows',
    'Maximum absolute difference', 'Mean absolute difference',
    'array_equal (bit-for-bit)', 'allclose (rtol=1e-05, atol=1e-08, NumPy defaults)',
])
match_summary

,Result
Total rows,1061
Exact matching rows,1061
Mismatching rows,0
Maximum absolute difference,0.0
Mean absolute difference,0.0
array_equal (bit-for-bit),PASS
"allclose (rtol=1e-05, atol=1e-08, NumPy defaults)",PASS


# 5. Manual Metric Recomputation

## 5.1 Metrics from the Independently Reconstructed Vector

> These metrics are computed from the independently reconstructed Naive vector, not from the stored `Naive` forecast column.

Uses the same canonical metric functions (`src.metrics.mae/rmse/mape/smape/mase`) as Notebook 07.

In [5]:
manual_metrics = pd.Series({
    'MAE': mae(test, naive),
    'RMSE': rmse(test, naive),
    'MAPE': mape(test, naive),
    'sMAPE': smape(test, naive),
    'MASE': mase(test, naive, train, 1),
}, name='Manual Reconstruction').to_frame().T
manual_metrics.round(3)

,MAE,RMSE,MAPE,sMAPE,MASE
Manual Reconstruction,1290.353,1853.625,1.743,1.744,4.576


## 5.2 Cross-Check Against the Authoritative Ranking

The authoritative-ranking metrics below are computed from the stored `Naive` column via the repository's canonical `metric_table` function -- the same source and function Notebook 07 uses for its full ranking.

In [6]:
authoritative_naive = metric_table(validated.Actual, {'Naive': validated.Naive}, train)[
    ['MAE', 'RMSE', 'MAPE', 'sMAPE', 'MASE']
].iloc[0]

cross_check = pd.DataFrame({
    'Manual Reconstruction': manual_metrics.iloc[0],
    'Authoritative Ranking (Notebook 07)': authoritative_naive,
})
cross_check['Absolute Difference'] = (cross_check['Manual Reconstruction'] - cross_check['Authoritative Ranking (Notebook 07)']).abs()
cross_check.round(6)

,Manual Reconstruction,Authoritative Ranking (Notebook 07),Absolute Difference
MAE,1290.353242,1290.353242,0.0
RMSE,1853.624774,1853.624774,0.0
MAPE,1.742747,1.742747,0.0
sMAPE,1.744142,1.744142,0.0
MASE,4.575633,4.575633,0.0


In [7]:
assert np.allclose(cross_check['Manual Reconstruction'], cross_check['Authoritative Ranking (Notebook 07)'])
print('All manual metrics match the authoritative ranking metrics.')

All manual metrics match the authoritative ranking metrics.


> The independently reconstructed Naive metrics agree with the Naive row in Notebook 07's authoritative full-model ranking.

# 6. Boundary and Identity Checks

### First-Test-Origin Check
Proves the first test forecast uses the final training observation. Failure would indicate a boundary shift or improper first-origin construction.

### Rolling Persistence Identity
Proves that, for every later test target, the forecast equals the immediately preceding revealed actual. Failure would indicate a shift bug, wrong lag, or accidental use of a different history value.

### Frozen-Vector Identity
Proves the independently reconstructed vector equals the authoritative stored Naive vector. Failure would indicate artifact drift or inconsistent forecast construction.

### Target Exclusion
Proves the manual forecast is not simply a copy of the same-day actual (the signature of an accidental zero-shift / leakage bug), since Bitcoin's daily Close is never exactly flat across the whole test period.

In [8]:
boundary_checks = pd.DataFrame([
    {'Check': 'First forecast boundary', 'Expected Relation': 'ManualNaive[0] == last_train_actual',
     'Result': 'PASS' if np.isclose(naive.iloc[0], train.iloc[-1]) else 'FAIL'},
    {'Check': 'Rolling lag-1 identity', 'Expected Relation': 'ManualNaive[t] == Actual[t-1] for all later t',
     'Result': 'PASS' if np.allclose(naive.iloc[1:], test.iloc[:-1]) else 'FAIL'},
    {'Check': 'Frozen vector identity', 'Expected Relation': 'ManualNaive == FrozenNaive (all 1,061 rows)',
     'Result': 'PASS' if np.allclose(naive.to_numpy(), validated.Naive.to_numpy()) else 'FAIL'},
    {'Check': 'Target excluded', 'Expected Relation': 'ManualNaive is not simply same-day Actual[t]',
     'Result': 'PASS' if not np.isclose(naive.to_numpy(), test.to_numpy()).all() else 'FAIL'},
]).set_index('Check')
assert (boundary_checks['Result'] == 'PASS').all()
boundary_checks

,Expected Relation,Result
Check,,
First forecast boundary,ManualNaive[0] == last_train_actual,PASS
Rolling lag-1 identity,ManualNaive[t] == Actual[t-1] for all later t,PASS
Frozen vector identity,"ManualNaive == FrozenNaive (all 1,061 rows)",PASS
Target excluded,ManualNaive is not simply same-day Actual[t],PASS


These checks establish that the implementation is correct and leakage-free at the reconstruction level audited here. They do not, by themselves, prove that Naive's strong ranking (Notebook 07) is explained solely by legitimate persistence rather than some other property of the series -- only that the forecast mechanism itself is exactly what it claims to be.

# 7. Key Findings

- The frozen `Naive` vector is exactly one-step persistence: `y_hat_t = y_(t-1)`, with no other transformation.
- The first final-test forecast uses the final training observation (Section 6, First-Test-Origin Check).
- Every later forecast uses only the immediately preceding revealed actual (Section 6, Rolling Persistence Identity).
- The independently reconstructed vector matches the frozen vector across all 1,061 rows: 1,061 exact matches, 0 mismatches, maximum absolute difference 0.0 (Section 4.2).
- Independently recomputed metrics (MAE 1290.353, RMSE 1853.625, MAPE 1.743, sMAPE 1.744, MASE 4.576) match Notebook 07's authoritative Naive ranking row exactly (Section 5.2).

> The strong Naive result is therefore attributable to genuine short-horizon persistence in the evaluated Bitcoin price series, not to same-day target leakage or an index-construction error.

This notebook verifies the implementation. It does not prove that persistence is the only reason Naive performs strongly -- only that the reported strength is not an artifact of a construction bug.

# 8. Limitations

### Train/Test Split
This notebook does not independently audit whether the canonical train/test split itself is leakage-free. That belongs to `01_Bitcoin_Data_EDA.ipynb` and the broader protocol documentation.

### Other Models
Only Naive receives this full independent re-derivation audit. The other nine forecast vectors are validated through their own model notebooks, artifact alignment checks, freeze validation (Notebook 07), and the repository verifier -- but are not independently reconstructed here.

### Frozen Target Assumption
This audit trusts the canonical `Actual` target vector loaded from the established data pipeline (`src.bitcoin_pipeline.load_bitcoin_target`).

### Numerical Equality
Frozen-vector identity in this notebook holds under both bit-for-bit `array_equal` and `allclose` (NumPy default tolerance: `rtol=1e-05`, `atol=1e-08`) -- the stronger bit-for-bit check passed, so the looser tolerance was not needed to establish agreement.

# 9. Next Notebook

Next: `09_Bitcoin_Robustness_and_Temporal_Stability.ipynb`

With the strongest simple baseline now independently validated, the workflow moves from point-forecast correctness to conditional behaviour: performance under training-defined volatility/movement regimes, and performance across earlier/middle/later test segments.